# 🏋️ Using Weights in Imbalanced Datasets (Higgs Boson Example)

This notebook shows how using weights can improve model performance on imbalanced data.

In [ ]:
# Install XGBoost if needed
# !pip install xgboost

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split


## 1️⃣ Create Toy Dataset

In [ ]:

# Simulate 1000 background events (label 0), 10 signal events (label 1)
n_background = 1000
n_signal = 10

# Create features: random numbers
X_background = np.random.normal(0, 1, (n_background, 2))  # background cluster
X_signal = np.random.normal(2, 1, (n_signal, 2))          # signal cluster (shifted)

# Stack the data
X = np.vstack((X_background, X_signal))
y = np.array([0]*n_background + [1]*n_signal)

# Convert to DataFrame for weights
df = pd.DataFrame(X, columns=['feature1', 'feature2'])
df['Label'] = y

# Define weight column:
# Background weight = 1
# Signal weight = 1/100
df['Weight'] = df['Label'].apply(lambda x: 1/100 if x == 1 else 1)

# Quick check:
df['Label'].value_counts(), df['Weight'].sum()


## 2️⃣ Train/Test Split

In [ ]:

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, df['Weight'].values, test_size=0.3, random_state=42, stratify=y)


## 3️⃣ Train WITHOUT Weights

In [ ]:

clf_no_weights = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
clf_no_weights.fit(X_train, y_train)

# Predict
y_pred_no_weights = clf_no_weights.predict(X_test)

# Evaluate
print("WITHOUT Weights:")
print(classification_report(y_test, y_pred_no_weights))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_no_weights))


## 4️⃣ Train WITH Weights

In [ ]:

clf_with_weights = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
clf_with_weights.fit(X_train, y_train, sample_weight=w_train)

# Predict
y_pred_with_weights = clf_with_weights.predict(X_test)

# Evaluate
print("WITH Weights:")
print(classification_report(y_test, y_pred_with_weights))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_with_weights))


## 5️⃣ Optional: Visualize Decision Boundary

In [ ]:

from matplotlib.colors import ListedColormap

def plot_decision_boundary(clf, X, y, title):
    x_min, x_max = X[:,0].min() - 1, X[:,0].max() + 1
    y_min, y_max = X[:,1].min() - 1, X[:,1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                         np.arange(y_min, y_max, 0.1))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(8,6))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(('blue', 'red')))
    plt.scatter(X[:,0], X[:,1], c=y, edgecolors='k', cmap=ListedColormap(('blue', 'red')))
    plt.title(title)
    plt.xlabel('feature1')
    plt.ylabel('feature2')
    plt.show()

# Plot both
plot_decision_boundary(clf_no_weights, X_test, y_test, "WITHOUT Weights")
plot_decision_boundary(clf_with_weights, X_test, y_test, "WITH Weights")
